In [1]:
%pip install -r HurrReqs.txt

Looking in indexes: https://pypi.org/simple, https://download.pytorch.org/whl/cu128
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import pandas as pd
import numpy as np
import duckdb
import warnings
warnings.filterwarnings("ignore")

nfip_df = pd.read_parquet("nfip_clean.parquet")
storm_df = pd.read_parquet("storm_events_clean.parquet")
hurdat2_df = pd.read_parquet("hurdat2_clean.parquet")   
nri_df = pd.read_parquet("nri_clean.parquet")

print(f"NFIP DataFrame shape: {nfip_df.shape}")
print(f"Storm DataFrame shape: {storm_df.shape}")
print(f"Hurdat2 DataFrame shape: {hurdat2_df.shape}")
print(f"NRI DataFrame shape: {nri_df.shape}")

NFIP DataFrame shape: (910075, 80)
Storm DataFrame shape: (3215, 55)
Hurdat2 DataFrame shape: (11222, 25)
NRI DataFrame shape: (85154, 12)


In [3]:
nfip_df['dateOfLoss'] = pd.to_datetime(nfip_df['dateOfLoss']).dt.tz_localize(None)

print(f"dateOfLoss sample: {nfip_df['dateOfLoss'].head(3).tolist()}")
print(f"dtype: {nfip_df['dateOfLoss'].dtype}")

dateOfLoss sample: [Timestamp('2008-09-12 00:00:00'), Timestamp('2017-08-27 00:00:00'), Timestamp('2008-09-13 00:00:00')]
dtype: datetime64[us]


In [4]:
con = duckdb.connect()
con.register("nfip", nfip_df)
con.register("storms", storm_df)

nfip_storm_df = con.execute("""
WITH ranked AS (
    SELECT 
        n.*,
        s.EPISODE_ID                  AS storm_episode_id,
        s.EVENT_ID                    AS storm_event_id,
        s.CATEGORY                    AS storm_category,
        s.DAMAGE_PROPERTY_CLEANED     AS storm_damage_property,
        s.BEGIN_DATE_TIME             AS storm_begin_date_time,
        s.END_DATE_TIME               AS storm_end_date_time,
        CASE s.EVENT_TYPE
            WHEN 'Hurricane'             THEN 1
            WHEN 'Hurricane (Typhoon)'   THEN 1
            WHEN 'Tropical Storm'        THEN 2
            WHEN 'Tropical Depression'   THEN 3
            WHEN 'Storm Surge/Tide'      THEN 4
            WHEN 'Coastal Flood'         THEN 5
            WHEN 'FLASH FLOOD'           THEN 6
            WHEN 'Flood'                 THEN 7
            ELSE 8
        END AS match_priority,
        ROW_NUMBER() OVER (
            PARTITION BY n.id
            ORDER BY 
                CASE s.EVENT_TYPE
                    WHEN 'Hurricane'             THEN 1
                    WHEN 'Hurricane (Typhoon)'   THEN 1
                    WHEN 'Tropical Storm'        THEN 2
                    WHEN 'Tropical Depression'   THEN 3
                    WHEN 'Storm Surge/Tide'      THEN 4
                    WHEN 'Coastal Flood'         THEN 5
                    WHEN 'FLASH FLOOD'           THEN 6
                    WHEN 'Flood'                 THEN 7
                    ELSE 8
                END ASC
        ) AS rn
    FROM nfip n
    LEFT JOIN storms s
        ON n.fips_code = s.FULL_FIPS
        AND n.dateOfLoss 
        BETWEEN s.BEGIN_DATE_TIME AND s.END_DATE_TIME + INTERVAL '30 day'
)
SELECT * EXCLUDE (rn, match_priority)
FROM ranked 
WHERE rn = 1
""").df()
print(f"rows after join: {nfip_storm_df.shape[0]}")
print(f"rows with storm data: {nfip_storm_df['storm_event_id'].notnull().sum()}")
print(f"rows without storm data: {nfip_storm_df['storm_event_id'].isnull().sum()}")
print(f"storm categories distribution:\n{nfip_storm_df['storm_category'].value_counts(dropna=False)}")

rows after join: 910075
rows with storm data: 120281
rows without storm data: 789794
storm categories distribution:
storm_category
NaN    910075
Name: count, dtype: int64


In [5]:
nfip_storm_df['has_storm_data'] = nfip_storm_df['storm_event_id'].notnull().astype(int)

print(f"Rows with storm data: {nfip_storm_df['has_storm_data'].sum()}")
print(f"Rows without storm data: {(nfip_storm_df['has_storm_data'] == 0).sum()}")


Rows with storm data: 120281
Rows without storm data: 789794


In [6]:
nfip_storm_df['storm_name_cleaned'] = (
    nfip_storm_df['floodEvent']
    .str.upper()
    .str.replace('HURRICANE ', '', regex=False)
    .str.replace('TROPICAL STORM ', '', regex=False)
    .str.replace('TROPICAL DEPRESSION ', '', regex=False)
    .str.replace(r'\s*\(.*?\)', '', regex=True)
    .str.strip()
)
hurdat2_peak = (
    hurdat2_df
    .sort_values(['maximum_sustained_wind_knots'], ascending=False)
    .groupby('storm_name')
    .first()
    .reset_index()[['storm_name', 'maximum_sustained_wind_knots', 'central_pressure_mb', 'status_of_system']]
    .rename(columns={
        'maximum_sustained_wind_knots': 'hurdat2_max_wind_speed',
        'central_pressure_mb': 'hurdat2_min_pressure_mb',
        'status_of_system': 'hurdat2_peak_status'
    })
    )
nfip_storm_df = nfip_storm_df.merge(
    hurdat2_peak,
    left_on='storm_name_cleaned',
    right_on='storm_name',
    how='left'
).drop(columns=['storm_name'])

matched = nfip_storm_df[nfip_storm_df['storm_event_id'].notna()]
print(f"Matched events with hurdat2 data: {matched['hurdat2_max_wind_speed'].notna().sum()}")
print(f"Matched events without hurdat2 data: {matched['hurdat2_max_wind_speed'].isna().sum()}")
print(f"Sample matched events with hurdat2 data:\n{matched[matched['hurdat2_max_wind_speed'].notna()][['floodEvent', 'storm_name_cleaned', 'hurdat2_max_wind_speed', 'hurdat2_min_pressure_mb', 'hurdat2_peak_status']].head(5)}")


Matched events with hurdat2 data: 120025
Matched events without hurdat2 data: 256
Sample matched events with hurdat2 data:
         floodEvent storm_name_cleaned  hurdat2_max_wind_speed  \
0  Hurricane Harvey             HARVEY                   115.0   
1  Hurricane Isaias             ISAIAS                    80.0   
2  Hurricane Harvey             HARVEY                   115.0   
3    Hurricane Irma               IRMA                   155.0   
4     Hurricane Ian                IAN                   140.0   

   hurdat2_min_pressure_mb hurdat2_peak_status  
0                    941.0                  HU  
1                    986.0                  HU  
2                    941.0                  HU  
3                    914.0                  HU  
4                    937.0                  HU  


In [7]:
def haversine_nm(lat1, lon1, lat2, lon2):
    R = 3440.065  # Earth radius in nautical miles
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = np.sin(dlat / 2)**2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2)**2
    c = 2 * np.arcsin(np.sqrt(a))
    return R * c

def get_bearing(lat1, lon1, lat2, lon2):
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    dlon = lon2 - lon1
    x = np.sin(dlon) * np.cos(lat2)
    y = np.cos(lat1) * np.sin(lat2) - np.sin(lat1) * np.cos(lat2) * np.cos(dlon)
    return (np.degrees(np.arctan2(x, y)) + 360) % 360

def get_quadrant(bearing):
    if (bearing >= 0 and bearing < 90):
        return 'ne'
    elif (bearing >= 90 and bearing < 180):
        return 'se'
    elif (bearing >= 180 and bearing < 270):
        return 'sw'
    else:
        return 'nw'

def wind_exposure_at_property( property_lat, property_lon, storm_observations):
    if len(storm_observations) == 0:
        return 0, np.nan
    min_dist = np.inf
    max_wind_threshold = 0
    for _, obs in storm_observations.iterrows():
        dist = haversine_nm(obs['latitude'], obs['longitude'], property_lat, property_lon)
        min_dist = min(min_dist, dist)
        bearing  = get_bearing(obs['latitude'], obs['longitude'], property_lat, property_lon)
        quad = get_quadrant(bearing)
        for kt in [64, 50, 34]:
            col = f'{kt}_kt_{quad}_nm'
            if col in obs.index and pd.notna(obs[col]) and obs[col] > 0:
                if dist <= obs[col]:
                    max_wind_threshold = max(max_wind_threshold, kt)
                    break
    return  max_wind_threshold, min_dist

hurdat2_by_storm = {
    name: group for name, group in hurdat2_df.groupby('storm_name')
}

unique_pairs = (
    nfip_storm_df[['latitude', 'longitude', 'storm_name_cleaned']]
    .drop_duplicates()
    .dropna(subset=['latitude', 'longitude', 'storm_name_cleaned'])
    .copy()
)

print(f"processing {len(unique_pairs)} unique property-storm pairs for wind exposure calculation")
results = []
for _, row in unique_pairs.iterrows():
    storm_observations = hurdat2_by_storm.get(row['storm_name_cleaned'], pd.DataFrame())
    wind_kt, dist_mn = wind_exposure_at_property(row['latitude'], row['longitude'], storm_observations)
    results.append({
        'latitude': row['latitude'],
        'longitude': row['longitude'],
        'storm_name_cleaned': row['storm_name_cleaned'],
        'prop_max_wind_kt': wind_kt,
        'prop_dist_to_track_nm': dist_mn
    })  
wind_df = pd.DataFrame(results)
    
nfip_storm_df = nfip_storm_df.merge(
    wind_df,    
    on=['latitude', 'longitude', 'storm_name_cleaned'],
    how='left'
)

print(f"\nWind threshold distribution:")
print(nfip_storm_df['prop_max_wind_kt'].value_counts().sort_index())
print(f"\nDistance to track (nm) summary:")
print(nfip_storm_df['prop_dist_to_track_nm'].describe())
print(f"\nProperties in hurricane force winds (≥64kt): {(nfip_storm_df['prop_max_wind_kt'] >= 64).sum():,}")
print(f"Properties in tropical storm force (≥34kt):  {(nfip_storm_df['prop_max_wind_kt'] >= 34).sum():,}")
print("\nWind interpolation complete ✅")


processing 18512 unique property-storm pairs for wind exposure calculation

Wind threshold distribution:
prop_max_wind_kt
0     174536
34    148789
50    268575
64    318175
Name: count, dtype: int64

Distance to track (nm) summary:
count    908573.000000
mean         67.936627
std          72.777639
min           0.000000
25%          23.884006
50%          42.349702
75%          92.707905
max        7167.109355
Name: prop_dist_to_track_nm, dtype: float64

Properties in hurricane force winds (≥64kt): 318,175
Properties in tropical storm force (≥34kt):  735,539

Wind interpolation complete ✅


In [8]:
if 'censusTract' in nfip_storm_df.columns:
    nfip_storm_df['censusTract'] = (
    nfip_storm_df['censusTract']
    .astype(str)
    .str.replace('.0', '', regex=False)
    .str.zfill(11)
)
nri_df['TRACTFIPS'] = (
    nri_df['TRACTFIPS']
    .astype(float)
    .astype(int)
    .astype(str)
    .str.zfill(11)
)

nfip_storm_df = nfip_storm_df.merge(
    nri_df,
    left_on='censusTract',
    right_on='TRACTFIPS',
    how='left'
)
merged_nri = nfip_storm_df['SOVI_SCORE'].notna().sum()


print(f"Total rows:            {len(nfip_storm_df):,}")
print(f"Rows with NRI data:    {merged_nri:,}")
print(f"Rows without NRI data: {len(nfip_storm_df) - merged_nri:,}")
print("\nNRI join complete ✅")
            

Total rows:            910,075
Rows with NRI data:    706,648
Rows without NRI data: 203,427

NRI join complete ✅


In [9]:
nfip_storm_df = nfip_storm_df.drop(columns=['TRACTFIPS'], errors='ignore')

print(f"Final DataFrame shape: {nfip_storm_df.shape}")

nfip_storm_df.to_parquet("nfip_joined.parquet", index=False)

print(f"nfip_joined.parquet {len(nfip_storm_df):,} rows, {len(nfip_storm_df.columns)} columns")
print("Data saved to nfip_joined.parquet ✅")

Final DataFrame shape: (910075, 104)
nfip_joined.parquet 910,075 rows, 104 columns
Data saved to nfip_joined.parquet ✅


feature engineering

In [10]:
nfip_raw = pd.read_parquet("nfip_clean.parquet")[['id', 'originalConstructionDate']].copy()
nfip_raw['construction_year'] = pd.to_datetime(nfip_raw['originalConstructionDate'], errors='coerce').dt.year

nfip_storm_df = nfip_storm_df.drop(columns=['originalConstructionDate', 'building_age_at_loss'], errors='ignore')
nfip_storm_df = nfip_storm_df.merge(
    nfip_raw[['id', 'construction_year']],
    on='id',
    how='left'
)

nfip_storm_df['building_age_at_loss'] = (nfip_storm_df['dateOfLoss'].dt.year - nfip_storm_df['construction_year']).clip(0,150)

print(f"building_age_at_loss nulls: {nfip_storm_df['building_age_at_loss'].isna().sum()}")
print(f"Sample values: {nfip_storm_df['building_age_at_loss'].dropna().sample(5).tolist()}")

building_age_at_loss nulls: 39
Sample values: [81.0, 45.0, 51.0, 39.0, 58.0]


In [11]:
nfip_storm_df['building_age_at_loss']  = (
    nfip_storm_df['yearOfLoss'] - nfip_storm_df['construction_year']
).clip(0,150)

nfip_storm_df['building_coverage_ratio'] = np.where(
    nfip_storm_df['buildingPropertyValue'] > 0,
    nfip_storm_df['totalBuildingInsuranceCoverage'] / nfip_storm_df['buildingPropertyValue'],
    np.nan
).clip(0, 2)

nfip_storm_df['replacement_cost_ratio'] = np.where(
    nfip_storm_df['buildingReplacementCost'] > 0,
    nfip_storm_df['totalBuildingInsuranceCoverage'] / nfip_storm_df['buildingReplacementCost'],
    np.nan
).clip(0, 2)

def wind_category(kt):
    if pd.isna(kt):
        return 'unknown'
    if kt >= 137:
        return 'cat5'
    elif kt >= 113:
        return 'cat4'
    elif kt >= 96:
        return 'cat3'
    elif kt >= 83:
        return 'cat2'
    elif kt >= 64:
        return 'cat1'
    elif kt >= 34:
        return 'tropical_storm_strong'
    elif kt >= 0:
        return 'tropical_storm_weak'
    else:
        return 'ts'
    
nfip_storm_df['storm_intensity_cat'] = nfip_storm_df['hurdat2_max_wind_speed'].apply(wind_category)

nfip_storm_df['prop_wind_exposure'] = nfip_storm_df['prop_max_wind_kt'].map({
    0: 'no exposure',
    34: 'ts_force',
    50: 'strong_ts_force',
    64: 'hurricane_force'
}).fillna('no_exposure')

nfip_storm_df['prop_dist_to_track_nm_bucket'] = pd.cut(
    nfip_storm_df['prop_dist_to_track_nm'], bins=[0, 25, 50, 100, 250, 500, np.inf], labels=['0-25nm', '25-50nm', '50-100nm', '100-250nm', '250-500nm', '500+nm'])

nfip_storm_df['post_firm_binary'] = (
    nfip_storm_df['postFIRMConstructionIndicator']
    .astype(str).str.upper().str.strip()
    .map({'Y': 1, 'TRUE': 1, '1': 1, 'N': 0, 'FALSE': 0, '0': 0})
)

nfip_storm_df['elevated_binary'] = (
    nfip_storm_df['elevatedBuildingIndicator']
    .astype(str).str.upper().str.strip()
    .map({'Y': 1, 'TRUE': 1, '1': 1, 'N': 0, 'FALSE': 0, '0': 0})
)

HIGH_RISK_ZONES = ['A', 'AE', 'AH', 'AO', 'AR', 'A99', 'V', 'VE']
nfip_storm_df['high_risk_Flood_zone'] = flood_zone_clean = nfip_storm_df['ratedFloodZone'].fillna('').astype(str).str.upper().str.strip()
nfip_storm_df['high_risk_Flood_zone'] = flood_zone_clean.apply(
    lambda z: 1 if z and any(z.startswith(h) for h in HIGH_RISK_ZONES) else 0
)

engineered_cols = [
    'building_age_at_loss',
    'building_coverage_ratio',
    'replacement_cost_ratio',
    'storm_intensity_cat',
    'prop_wind_exposure',
    'prop_dist_to_track_nm_bucket',
    'post_firm_binary',
    'elevated_binary',
    'high_risk_Flood_zone'
]

for col in engineered_cols:
    null_count = nfip_storm_df[col].isna().sum()
    print(f" {col:>35} nulls {null_count:,}")
    
print("Feature engineering complete ✅")

                building_age_at_loss nulls 39
             building_coverage_ratio nulls 12,170
              replacement_cost_ratio nulls 26,086
                 storm_intensity_cat nulls 0
                  prop_wind_exposure nulls 0
        prop_dist_to_track_nm_bucket nulls 5,484
                    post_firm_binary nulls 0
                     elevated_binary nulls 0
                high_risk_Flood_zone nulls 0
Feature engineering complete ✅


In [12]:
PROPERTY_FEATURES = [
    'basementEnclosureCrawlspaceType',
    'numberOfFloorsInTheInsuredBuilding',
    'occupancyType',
    'obstructionType',
    'buildingDescriptionCode',
    'numberOfUnits',
    'locationOfContents',
    'elevationDifference',
    'lowestAdjacentGrade',
    'baseFloodElevation',
    'ratedFloodZone',
    'floodZoneCurrent',
    'high_risk_Flood_zone',
    'building_age_at_loss',
    'post_firm_binary',
    'elevated_binary',
    'floodproofedIndicator',
    'totalBuildingInsuranceCoverage',
    'buildingPropertyValue',
    'buildingReplacementCost',
    'building_coverage_ratio',
    'replacement_cost_ratio',
    'primaryResidenceIndicator',
    'rentalPropertyIndicator',
    'houseWorship',
    'agricultureStructureIndicator',
    'nonProfitIndicator',
    'smallBusinessIndicatorBuilding',
    'stateOwnedIndicator',
    'policyCount',
    'crsClassificationCode'
]

STORM_FEATURES = [
    'hurdat2_max_wind_speed',
    'hurdat2_min_pressure_mb',
    'prop_max_wind_kt',
    'prop_dist_to_track_nm',
    'causeOfDamage',
    'storm_intensity_cat',
    'prop_wind_exposure',
    'prop_dist_to_track_nm_bucket'
]

NRI_FEATURES = [
    'CFLD_AFREQ',
    'CFLD_EALB',
    'CFLD_ALRB',
    'SOVI_SCORE',
    'RESL_SCORE',
    'RISK_SCORE'
]

GEO_FEATURES = [
    'state',
    'latitude',
    'longitude',
    'yearOfLoss'
]

ALL_FEATURES = PROPERTY_FEATURES + STORM_FEATURES + NRI_FEATURES + GEO_FEATURES
TARGET = 'buildingDamageAmount_2024'

missing = [f for f in ALL_FEATURES if f not in nfip_storm_df.columns]
present = [f for f in ALL_FEATURES if f in nfip_storm_df.columns]
print(f"Features missing from DataFrame: {len(missing)}")
print(f"Features present in DataFrame: {len(present)}")

model_df = nfip_storm_df[ALL_FEATURES + [TARGET, 'yearOfLoss', 'floodEvent', 'id', 'lambda', 'low_data_flag']].copy()
model_df = model_df[model_df[TARGET] > 0].copy()

print(f"\nModeling rows (damage > 0): {len(model_df):,}")
print(f"Target distribution:")
print(model_df[TARGET].describe().apply(lambda x: f"${x:,.0f}"))
print("\nFeature set complete ✅")


Features missing from DataFrame: 0
Features present in DataFrame: 49

Modeling rows (damage > 0): 910,075
Target distribution:
count          $910,075
mean            $83,108
std          $1,259,935
min                  $1
25%             $11,507
50%             $44,457
75%            $108,454
max      $1,083,342,000
Name: buildingDamageAmount_2024, dtype: str

Feature set complete ✅


In [13]:
cap = model_df[TARGET].quantile(0.999)
print(f"Applying cap at 99.9th percentile: ${cap:,.0f}")

model_df['buildingDamageAmount_2024_capped'] = model_df['buildingDamageAmount_2024'].clip(upper=cap)

TARGET = 'buildingDamageAmount_2024_capped'

print(f"\nCapped target distribution:")
print(model_df[TARGET].describe().apply(lambda x: f"${x:,.0f}"))


Applying cap at 99.9th percentile: $1,415,065

Capped target distribution:
count      $910,075
mean        $79,768
std        $111,122
min              $1
25%         $11,507
50%         $44,457
75%        $108,454
max      $1,415,065
Name: buildingDamageAmount_2024_capped, dtype: str


In [14]:
model_df = model_df.loc[:, ~model_df.columns.duplicated()].copy()
print(f"\nFinal model DataFrame shape: {model_df.shape[1]}")


Final model DataFrame shape: 55


In [17]:
from sklearn.preprocessing import LabelEncoder

cat_features = [f for f in ALL_FEATURES if model_df[f].dtype == 'object' or str(model_df[f].dtype) == 'category']

num_features = [f for f in ALL_FEATURES if f not in cat_features]

print(f"Categorical features {len(cat_features)}")
print(f"Numerical features {len(num_features)}")
print(f"categorical features: {cat_features}")
label_encoders = {}
for col in cat_features:
    le = LabelEncoder()
    le.fit(model_df[col].astype(str).fillna('UNKNOWN'))
    model_df[col] = le.transform(model_df[col].astype(str).fillna('UNKNOWN')).astype(int)
    label_encoders[col] = le
    
print("Label encoding complete ✅")
print(f"final model DataFrame shape: {model_df.shape}")
print(f"Null count in target: {model_df[TARGET].isna().sum()}")


Categorical features 1
Numerical features 48
categorical features: ['prop_dist_to_track_nm_bucket']
Label encoding complete ✅
final model DataFrame shape: (910075, 55)
Null count in target: 0


In [19]:
TEST_STORMS = ['Hurricane Ivan', 'Hurricane Sandy', 'Hurricane Ian']

train_df = model_df[~model_df['floodEvent'].isin(TEST_STORMS)].copy()
test_df = model_df[model_df['floodEvent'].isin(TEST_STORMS)].copy()

X_train = train_df[ALL_FEATURES]
y_train = train_df[TARGET]

X_test = test_df[ALL_FEATURES]
y_test = test_df[TARGET]

print(f"Train rows: {len(X_train):,}  ({len(train_df['floodEvent'].unique())} storms)")
print(f"Test rows:  {len(X_test):,}  ({len(test_df['floodEvent'].unique())} storms)")
print(f"\nTest storms:")
print(test_df['floodEvent'].value_counts())

train_df.to_parquet("train_df.parquet", index=False)
test_df.to_parquet("test_df.parquet", index=False)

import pickle
with open("feature_list.pkl", "wb") as f:
    pickle.dump({'ALL_FEATURES': ALL_FEATURES, 'TARGET': TARGET}, f)

with open("label_encoder.pkl", 'wb') as f:
    pickle.dump(label_encoders, f)
print("\nTrain/test data saved ✅")

Train rows: 720,526  (77 storms)
Test rows:  189,549  (3 storms)

Test storms:
floodEvent
Hurricane Sandy    135455
Hurricane Ian       38035
Hurricane Ivan      16059
Name: count, dtype: int64

Train/test data saved ✅
